In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import seaborn as sns
sns.set(style='whitegrid')
from stats_utils import aggregate_across_seeds

parameterizations = [
    'sp',
    # 'sp_with_mup_hidden_init',
    # 'sp_with_mup_hidden_init_and_lr',
    # 'sp_with_mup_hidden_init_and_lr_partial_output_logits',
    # 'sp_with_mup_hidden_init_and_lr_output_logits',
    # 'mup',
]

df_dict = {} # {parameterization: {job_name: df}}
for parameterization in parameterizations:
    df_dict[parameterization] = {}
    if parameterization == 'plot.ipynb':
        continue

    for job_name in os.listdir(os.path.join(parameterization, 'out')):
        try:
            df_dict[parameterization][job_name] = pd.read_csv(os.path.join(parameterization, 'out', job_name, 'log.csv'))
        except:
            pass

class MplColorHelper:

    def __init__(self, cmap_name, start_val, stop_val):
        self.cmap_name = cmap_name
        self.cmap = plt.get_cmap(cmap_name)
        self.norm = mpl.colors.Normalize(vmin=start_val, vmax=stop_val)
        self.scalarMap = cm.ScalarMappable(norm=self.norm, cmap=self.cmap)

    def get_rgb(self, val):
        return self.scalarMap.to_rgba(val)

layer_types = [
    ('token_embedding_act_abs_mean', 'Word Embedding', (1.5e-2, 3.5e-2)),
    ('attn_act_abs_mean', 'Attention Output', (1e-2, 1e4)),
    ('mlp_act_abs_mean', 'FFN Output', (1e-2, 1e4)),
    ('lm_head_act_abs_mean', 'Output Logits', (5e-2, 1e2)),
    ('last_layer_act_abs_mean', 'Last Layer Output', (1e-2, 1e4)),
]
parameterizations = [
    'sp',
    'sp_with_mup_hidden_init',
    'sp_with_mup_hidden_init_and_lr',
    'sp_with_mup_hidden_init_and_lr_partial_output_logits',
    'sp_with_mup_hidden_init_and_lr_output_logits',
    'mup',
]
seeds = [1,2,3,4,5]
widths = [256,512,1024,2048,4096]
# t_max = 1
# t_max = 2
t_max = 10

color_helper = MplColorHelper('coolwarm', 0, t_max)
n_cols = len(layer_types)
n_rows = len(parameterizations)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))

for parameterization_idx, parameterization in enumerate(parameterizations):
    results_matrix = np.zeros((len(layer_types), t_max, len(widths), len(seeds))) # (layer_type, t, width, seed)
    for width_idx, width in enumerate(widths):
        for seed_idx, seed in enumerate(seeds):
            width_str = float(width) if width != 0 else int(width)
            job_name = f'width{width}_depth2_seed{seed}'
            try:
                ckpt_df = df_dict[parameterization][job_name]
            except:
                print(job_name)
            if len(ckpt_df) == 0 or ckpt_df['step'].max() == 0:
                print(job_name)
                continue
            for layer_type_idx, (layer_type, layer_type_str, ylims) in enumerate(layer_types):
                results_matrix[layer_type_idx, :, width_idx, seed_idx] = ckpt_df[layer_type].dropna().values[:t_max].flatten()
                

    for layer_type_idx, (layer_type, layer_type_str, ylims) in enumerate(layer_types):
        ax = axes[parameterization_idx, layer_type_idx]
        for t in range(0,t_max):
            means = []
            stderrs = []
            for width_idx, width in enumerate(widths):
                nnz_results = results_matrix[layer_type_idx, t, width_idx][results_matrix[layer_type_idx, t, width_idx] != 0]
                means.append(nnz_results.mean())
                stderrs.append(np.std(nnz_results, ddof=1) / np.sqrt(len(nnz_results)))
            means = np.array(means)
            stderrs = np.array(stderrs)
            ax.plot(widths, means, label=f'{t+1}', color=color_helper.get_rgb(t), marker='.')
            ax.fill_between(widths, means-stderrs, means+stderrs, color=color_helper.get_rgb(t), alpha=0.5)

        ax.set_title(layer_type_str)
        ax.set_xlabel('Width')
        # axes[parameterization_idx, 0].set_ylabel(parameterization)
        axes[parameterization_idx, 0].set_ylabel('np.abs(activation).mean()')
        axes[parameterization_idx, 0].legend(loc='upper left', fontsize=8, title='Step')
        ax.set_xscale('log', base=2)
        ax.set_yscale('log')
        # ax.set_ylim(*ylims)
        # axes[layer_type_idx].set_ylim(1e-3, 1e3)
        # if variant_idx < n_rows-1:
        #     axes[layer_type_idx].xaxis.set_ticklabels([])

# plt.suptitle(r'Coordinate Check')
plt.tight_layout()
plt.show()
plt.close()

ModuleNotFoundError: No module named 'stats_utils'

In [ ]:
results_matrix.shape


(4, 10, 5, 5)

In [ ]:


def plot_metric(metric, with_error=True):
    metric_names = [(f'last_layer_act_abs_{metric}', 'Final Residual Stream', (1e-2, 1e2))]
    color_helper = MplColorHelper('coolwarm', 0, t_max)
    n_cols = len(layer_types)
    n_rows = len(parameterizations)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
    axes = np.array(axes).reshape((n_rows, n_cols))
    for p_idx, param in enumerate(parameterizations):
        results = np.zeros((len(metric_names), t_max, len(widths), len(seeds)))
        errs = np.zeros_like(results)
        for w_idx, width in enumerate(widths):
            for s_idx, seed in enumerate(seeds):
                job_name = f'width{width}_depth2_seed{seed}'
                df = df_dict[param].get(job_name)
                if df is None or len(df)==0:
                    continue
                for l_idx, (col, _, _) in enumerate(metric_names):
                    vals = df[col].dropna().values[:t_max]
                    results[l_idx, :len(vals), w_idx, s_idx] = vals
                    if with_error and metric=='mean':
                        e_col = col.replace('mean','std')
                        err_vals = df[e_col].dropna().values[:t_max]
                        errs[l_idx, :len(err_vals), w_idx, s_idx] = err_vals
        for l_idx, (col, title, ylims) in enumerate(metric_names):
            ax = axes[p_idx, l_idx]
            for t in range(t_max):
                ms = []
                es = []
                for w_idx, width in enumerate(widths):
                    vals = results[l_idx, t, w_idx]
                    err_v = errs[l_idx, t, w_idx]
                    mask = vals != 0
                    if not np.any(mask):
                        continue
                    if with_error:
                        m, e = aggregate_across_seeds(vals[mask], err_v[mask])
                    else:
                        m = vals[mask].mean()
                        e = vals[mask].std(ddof=1)/np.sqrt(mask.sum())
                    ms.append(m)
                    es.append(e)
                ms = np.array(ms)
                es = np.array(es)
                ax.plot(widths[:len(ms)], ms, label=f'{t+1}', color=color_helper.get_rgb(t), marker='.')
                ax.fill_between(widths[:len(ms)], ms-es, ms+es, color=color_helper.get_rgb(t), alpha=0.5)
            ax.set_title(title)
            ax.set_xlabel('Width')
            ax.set_xscale('log', base=2)
            ax.set_yscale('log')
            ax.set_ylim(*ylims)
            if l_idx>0:
                ax.yaxis.set_ticklabels([])
    axes[0,0].set_ylabel('|activation|')
    axes[0,0].legend(loc='upper left', fontsize=8, title='Step')
    plt.tight_layout()
    plt.show()
    plt.close()

plot_metric('min', with_error=False)
plot_metric('mean', with_error=False)
plot_metric('max', with_error=False)
plot_metric('mean', with_error=True)


ModuleNotFoundError: No module named 'parametrisation_examples'